<a href="https://colab.research.google.com/github/Maria-apsguiar/AD_OrchidLog/blob/main/CaseOrchidLog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
entrega_orquideas = '/content/entregas_orquideas.csv'
df_entrega_orquideas = pd.read_csv(entrega_orquideas)
print('Entrega Orquideas')
display(df_entrega_orquideas.head())

print('\n\n Estufas Qualidade')
estufas_qualidade = '/content/estufas_qualidade.csv'
df_estufas_qualidade = pd.read_csv(estufas_qualidade)
display(df_estufas_qualidade.head(10))

print('\n\n Telemetria')
telemetria_iot = '/content/telemetria_iot.csv'
df_telemetria_iot = pd.read_csv(telemetria_iot)
display(df_telemetria_iot.head())

Entrega Orquideas


,ID_Carga,Placa_Caminhao,Estufa_Origem,UF_Destino,Qtd_Orquideas,Status_Entrega
0,LOG-1000,TX-103,EST-3,SP,457,Entregue
1,LOG-1001,TX-100,EST-1,SC,165,Entregue
2,LOG-1002,TX-108,EST-2,MG,124,Entregue
3,LOG-1003,TX-107,EST-1,RJ,162,Entregue
4,LOG-1004,TX-107,EST-2,SP,469,Entregue




 Estufas Qualidade


,id_estufa,especie_predominante,umidade_media_pct,responsavel_tecnico,indice_qualidade_lote
0,EST-1,Phalaenopsis,75,Ana Silva,0.95
1,EST-2,Cattleya,82,Carlos Mendes,0.88
2,EST-3,Dendrobium,65,Mariana Costa,0.91




 Telemetria


,Placa_Caminhao,Temperatura_C,Alerta_Calor
0,TX-103,20,NAO
1,TX-100,30,SIM
2,TX-108,28,SIM
3,TX-107,33,SIM
4,TX-104,20,NAO


In [ ]:
list_status_temperatura = []
list_status = df_telemetria_iot['Temperatura_C'].tolist()

for i in list_status:
  if i >=  28:
    list_status_temperatura.append('ALERTA')
  else:
    list_status_temperatura.append('OK')

df_telemetria_iot['status'] = list_status_temperatura
display(df_telemetria_iot.head())


,Placa_Caminhao,Temperatura_C,Alerta_Calor,status
0,TX-103,20,NAO,OK
1,TX-100,30,SIM,ALERTA
2,TX-108,28,SIM,ALERTA
3,TX-107,33,SIM,ALERTA
4,TX-104,20,NAO,OK


In [ ]:
#Diagnóstico geral: Qual foi a quantidade de orquídeas transportadas em toda a base de dados?
soma_orquideas = df_entrega_orquideas['Qtd_Orquideas'].sum()
print(f'A quantidade de orquídeas transportadas em toda a base de dados foi de: {soma_orquideas}')
#soma

A quantidade de orquídeas transportadas em toda a base de dados foi de: 55425


In [ ]:
#Qual foi a quantidade total de orquideas perdidas:
qtd_perdidas = df_entrega_orquideas[df_entrega_orquideas['Status_Entrega'] =='Cancelado']['Qtd_Orquideas'].sum()
print(f'A quantidade total de orquideas perdidas foi de: {qtd_perdidas}')

A quantidade total de orquideas perdidas foi de: 6349


In [ ]:
#Impacto regional - Filtre apenas as canceladas. Qual foi o estado (UF_Destino) que mais apareceu nessa lista de perdas?
df_entrega_orquideas_canceladas = df_entrega_orquideas[df_entrega_orquideas['Status_Entrega'] =='Cancelado']
print(f"O Estado que mais apareceu na lista de perdas foi: {df_entrega_orquideas_canceladas['UF_Destino'].mode()[0]}")

O Estado que mais apareceu na lista de perdas foi: SP


In [ ]:
#Analise de temperatura: Qual foi a temperatura média registrada pelos caminhões?
temperatura_media = df_telemetria_iot['Temperatura_C'].mean()
print(f'A média dos pedidos de orquídeas foi de: {temperatura_media}')



A média dos pedidos de orquídeas foi de: 27.0


In [ ]:
#Média de pedidos de Orquideas
media_orquideas = df_entrega_orquideas['Qtd_Orquideas'].mean()
print(f'A média dos pedidos de orquídeas foi de: {media_orquideas}')

#media

A média dos pedidos de orquídeas foi de: 277.125


In [ ]:
#Qual estufa de origem teve maior indice de percas?
estufa_maior_perda = df_entrega_orquideas_canceladas['Estufa_Origem'].value_counts().idxmax()
print(f"A estufa de origem com o maior índice de perdas foi: {estufa_maior_perda}")

A estufa de origem com o maior índice de perdas foi: EST-2


In [ ]:
#Tivemos um pedido com soma de [soma_orquideas] e perdemos [qtd_perdidas], qual porcentagem representam as percas?
porcentagem_percas = (qtd_perdidas / soma_orquideas) * 100
print(f"A porcentagem de orquídeas perdidas é de: {porcentagem_percas:.2f}%")

A porcentagem de orquídeas perdidas é de: 11.46%


In [ ]:
#taxa de cancelamento por estufas
df_cancelamentos_por_estufa = df_entrega_orquideas.groupby('Estufa_Origem')['Status_Entrega'].value_counts().unstack(fill_value=0)
df_cancelamentos_por_estufa['Total_Pedidos'] = df_cancelamentos_por_estufa.sum(axis=1)
df_cancelamentos_por_estufa['Taxa_Cancelamento'] = (df_cancelamentos_por_estufa['Cancelado'] / df_cancelamentos_por_estufa['Total_Pedidos']) * 100

display(df_cancelamentos_por_estufa[['Cancelado', 'Total_Pedidos', 'Taxa_Cancelamento']].sort_values(by='Taxa_Cancelamento', ascending=False))

Status_Entrega,Cancelado,Total_Pedidos,Taxa_Cancelamento
Estufa_Origem,,,
EST-2,12,68,17.647059
EST-1,10,65,15.384615
EST-3,5,67,7.462687


In [ ]:
# Percentual de perdas de orquídeas (quantidade) por estufa
df_perdas_quantidade_por_estufa = df_entrega_orquideas.groupby('Estufa_Origem').agg(
    Total_Orquideas=('Qtd_Orquideas', 'sum'),
    Orquideas_Perdidas=('Qtd_Orquideas', lambda x: x[df_entrega_orquideas.loc[x.index, 'Status_Entrega'] == 'Cancelado'].sum())
).reset_index()

df_perdas_quantidade_por_estufa['Percentual_Perdas_Qtd'] = (df_perdas_quantidade_por_estufa['Orquideas_Perdidas'] / df_perdas_quantidade_por_estufa['Total_Orquideas']) * 100

display(df_perdas_quantidade_por_estufa[['Estufa_Origem', 'Orquideas_Perdidas', 'Total_Orquideas', 'Percentual_Perdas_Qtd']].sort_values(by='Percentual_Perdas_Qtd', ascending=False))

,Estufa_Origem,Orquideas_Perdidas,Total_Orquideas,Percentual_Perdas_Qtd
0,EST-1,2591,18170,14.259769
1,EST-2,2306,17247,13.370441
2,EST-3,1452,20008,7.257097
